In [1]:
import pandas as pd


In [4]:
Base_geral = (pd.read_csv('/content/Base_geral (1).csv'))
base_professores = pd.read_csv('/content/base_professores (2).csv')
datas_formacao_profs = pd.read_csv('/content/datas_formacao_profs.csv')
base_doutorandos = pd.read_csv('/content/base_doutorandos (2).csv')
datas_formacao_docs = pd.read_csv('/content/datas_formacao_docs.csv')
base_posdocs = (pd.read_csv('/content/base_posdocs (2).csv')).rename(columns={'Inicio':'Início'})
datas_formacao_posdocs = pd.read_csv('/content/datas_formacao_posdocs.csv')


In [5]:
Datas = pd.concat(
    [datas_formacao_profs, datas_formacao_docs, datas_formacao_posdocs],
    ignore_index=True
)
Base_geral = Base_geral.merge(Datas, how='inner', on='Pesquisador')

In [6]:
#periodo de publicação na graduação
df_graduandos = Base_geral[
    (Base_geral['Ano de Publicação'] >= Base_geral['Inicio GRADUACAO']) &
    (Base_geral['Ano de Publicação'] <= Base_geral['Fim GRADUACAO'])]
n_public_graduandos = df_graduandos['Pesquisador'].value_counts()
Docs_Public_grad = n_public_graduandos[n_public_graduandos.index.isin(base_doutorandos['Pesquisador'])]
Pos_docs_public_grad = n_public_graduandos[n_public_graduandos.index.isin(base_posdocs['Pesquisador'])]
Professores_public_grad = n_public_graduandos[n_public_graduandos.index.isin(base_professores['Pesquisador'])]
#periodo de publicação no mestrado
df_pesquisadores = Base_geral
df_pesquisadores['Teste para mestrado'] = df_pesquisadores['Ano de Publicação'].ge(df_pesquisadores['Inicio MESTRADO']) & df_pesquisadores['Ano de Publicação'].le(df_pesquisadores['Fim MESTRADO'])
df_pesquisadores = df_pesquisadores.where(df_pesquisadores['Teste para mestrado'] == True).dropna(how ='all')
N_que_publicou= df_pesquisadores['Pesquisador'].value_counts()
N_que_publicou_doutorandos = N_que_publicou[N_que_publicou.index.isin(base_doutorandos['Pesquisador'])]
N_que_publicou_posdocs = N_que_publicou[N_que_publicou.index.isin(base_posdocs['Pesquisador'])]
N_que_publicou_profs = N_que_publicou[N_que_publicou.index.isin(base_professores['Pesquisador'])]

#Período de publicação antes do vinculo ao IB
df_antes_do_vinculo = Base_geral[
    Base_geral['Ano de Publicação'] < Base_geral['Ínicio']
].copy()

Docs_Public_antes_vinculo = (
    df_antes_do_vinculo[
        df_antes_do_vinculo['Pesquisador'].isin(base_doutorandos['Pesquisador'])
    ])

PosDocs_Public_antes_vinculo = (
    df_antes_do_vinculo[
        df_antes_do_vinculo['Pesquisador'].isin(base_posdocs['Pesquisador'])
    ])

Professores_Public_antes_vinculo = (
    df_antes_do_vinculo[
        df_antes_do_vinculo['Pesquisador'].isin(base_professores['Pesquisador'])
    ])


#periodo de publicação após o vínculo com o IB

df_apos_do_vinculo = Base_geral[
    Base_geral['Ano de Publicação'] >= Base_geral['Ínicio']
].copy()

Professores_Public_apos_vinculo = (
    df_apos_do_vinculo[
        df_apos_do_vinculo['Pesquisador'].isin(base_professores['Pesquisador'])])

Docs_Public_apos_vinculo = (
    df_apos_do_vinculo[
        df_apos_do_vinculo['Pesquisador'].isin(base_doutorandos['Pesquisador'])])

PosDocs_Public_apos_vinculo = (
    df_apos_do_vinculo[
        df_apos_do_vinculo['Pesquisador'].isin(base_posdocs['Pesquisador'])])

#                      Diferença entre a média antes e após a afiliação (teste t)
from scipy.stats import ttest_rel

#teste t doutorandos
antes_docs = Docs_Public_antes_vinculo['Pesquisador'].value_counts()
depois_docs = Docs_Public_apos_vinculo['Pesquisador'].value_counts()

comparacao_docs = pd.concat(
    [antes_docs.rename('antes'),
     depois_docs.rename('depois')],
    axis=1
).fillna(0)


tdocs, pdocs = ttest_rel(
    comparacao_docs['depois'],
    comparacao_docs['antes']
)

#teste t pós doutorandos
antes_Pos_docs = PosDocs_Public_antes_vinculo['Pesquisador'].value_counts()
depois_Pos_docs = PosDocs_Public_apos_vinculo['Pesquisador'].value_counts()

comparacao_Posdocs = pd.concat(
    [antes_Pos_docs.rename('antes'),
     depois_Pos_docs.rename('depois')],
    axis=1
).fillna(0)


tPosdocs, pPosdocs = ttest_rel(
    comparacao_Posdocs['depois'],
    comparacao_Posdocs['antes']
)

#teste t professores
antes_profs = Professores_Public_antes_vinculo['Pesquisador'].value_counts()
depois_profs = Professores_Public_apos_vinculo['Pesquisador'].value_counts()

comparacao_profs = pd.concat(
    [antes_profs.rename('antes'),
     depois_profs.rename('depois')],
    axis=1
).fillna(0)


tprofs, pprofs = ttest_rel(
    comparacao_profs['depois'],
    comparacao_profs['antes']
)

#                     Calculando o tempo médio de publicação após vínculo ao IB-USP

#Tempo médio de publicação após vinculo para os professores
primeiro_artigoProfs = (
    Professores_Public_apos_vinculo
    .groupby('Pesquisador')['Ano de Publicação']
    .min()
    .reset_index()
    .rename(columns={'Ano de Publicação': 'PrimeiroArtigo'}))
inicioProfs = (
    Base_geral[['Pesquisador', 'Ínicio']]
    .drop_duplicates()
)

tempoProfs = primeiro_artigoProfs.merge(
    inicioProfs,
    on='Pesquisador',
    how='left'
)

tempoProfs['TempoPrimeiraPublicacao'] = (
    tempoProfs['PrimeiroArtigo'] - tempoProfs['Ínicio']
)
tempo_medioProfs = tempoProfs['TempoPrimeiraPublicacao'].mean()

#Tempo médio de publicação após vinculo para os doutorandos
primeiro_artigoDocs = (
    Docs_Public_apos_vinculo
    .groupby('Pesquisador')['Ano de Publicação']
    .min()
    .reset_index()
    .rename(columns={'Ano de Publicação': 'PrimeiroArtigo'}))
inicioDocs = (
    Base_geral[['Pesquisador', 'Ínicio']]
    .drop_duplicates()
)

tempoDocs = primeiro_artigoDocs.merge(
    inicioDocs,
    on='Pesquisador',
    how='left'
)

tempoDocs['TempoPrimeiraPublicacao'] = (
    tempoDocs['PrimeiroArtigo'] - tempoDocs['Ínicio']
)
tempo_medioDocs = tempoDocs['TempoPrimeiraPublicacao'].mean()


#Tempo médio de publicação após vinculo para os Pós-doutorandos
primeiro_artigoPosDocs = (
    PosDocs_Public_apos_vinculo
    .groupby('Pesquisador')['Ano de Publicação']
    .min()
    .reset_index()
    .rename(columns={'Ano de Publicação': 'PrimeiroArtigo'}))

inicioPosDocs = (
    Base_geral[['Pesquisador', 'Ínicio']]
    .drop_duplicates())
tempoPosDocs = primeiro_artigoPosDocs.merge(
    inicioDocs,
    on='Pesquisador',
    how='left')

tempoPosDocs['TempoPrimeiraPublicacao'] = (
    tempoPosDocs['PrimeiroArtigo'] - tempoPosDocs['Ínicio']
)
tempo_medioPosDocs = tempoPosDocs['TempoPrimeiraPublicacao'].mean()

#ENCONTRANDO A FREQUÊNCIA DE PUBLICAÇÃO ANUAL APÓS VINCULO AO IB
#para professores
#posterirmente substituir "professores_apos_vinculo" pelo correspondente aos pós-docs e doutorandos
artigos = Professores_Public_apos_vinculo['Pesquisador'].value_counts().reset_index()
artigos.columns = ['Pesquisador', 'Artigos']
inicio = Professores_Public_apos_vinculo[['Pesquisador', 'Ínicio']].drop_duplicates()
new_df = artigos.merge(inicio, on='Pesquisador', how='inner')
new_df['Tempo'] = 2022 - new_df['Ínicio']
new_df = new_df[new_df['Tempo'] > 0]
new_df['Frequência publicacao'] = new_df['Artigos'] / new_df['Tempo']
freqPublic = new_df['Frequência publicacao'].mean()

In [9]:
media_antes_docs = comparacao_docs['antes'].mean()
media_depois_docs = comparacao_docs['depois'].mean()

aumento_docs = (
    (media_depois_docs - media_antes_docs)
    / media_antes_docs
) * 100
aumento_docs

np.float64(49.27953890489912)

In [10]:
media_antes_pos = comparacao_Posdocs['antes'].mean()
media_depois_pos = comparacao_Posdocs['depois'].mean()

aumento_pos = (
    (media_depois_pos - media_antes_pos)
    / media_antes_pos
) * 100
aumento_pos

np.float64(50.74626865671642)

In [12]:
media_antes_prof = comparacao_profs['antes'].mean()
media_depois_prof = comparacao_profs['depois'].mean()

aumento_prof = (
    (media_depois_prof - media_antes_prof)
    / media_antes_prof
) * 100
aumento_prof

np.float64(86.352301065518)